In [1]:
%load_ext autoreload
%autoreload 2

check ว่า cosine สามารถใช้กับ centroid ได้ไหม

เอา mean ใช้ได้ไหม

วิธีไหนที่บอกได้ว่าเขามีความต้องการเรื่องนี้

input > query >

input > nextaction >

input > no intent >

input > query and nextaction >

เราจะใช้วิธีไหนดักว่า input คืออะไร

มองภาพ input output เป็นไง process ค่อยหาวิธีทำ

มองวิธีดัก เช่น ถ้าไม่มี intent เด้งออก

In [2]:
from package.llms.ollama import OllamaLLM, OpenAIOutputMessage, LlamaOutputMessage
from package.llms.bedrock import BedrockNova
# import logging
# from package.utils import setup_logger
from package.prompt_hub import PromptHub

# setup_logger(logging.DEBUG)

# openai_llm = OllamaLLM(model_id="gpt-oss:20b", OutputMessage=OpenAIOutputMessage)
# llama_llm = OllamaLLM(model_id="llama3.2", OutputMessage=LlamaOutputMessage)
nova_llm = BedrockNova(model_id="us.amazon.nova-micro-v1:0")

In [3]:
from package.intent_hub.query import query_general
from package.intent_hub.nextaction import nextaction

from package.agents.intent_agent import MultiIntentClassifier

model = MultiIntentClassifier()
model.add_examples(intent="query", examples=query_general)
# model.add_examples(intent="nextaction", examples=nextaction)

d:\Git\leonidas-arthena\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import numpy as np
from collections import defaultdict
from sentence_transformers import SentenceTransformer

# Global model instance
emb_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

def embed_text(text: str) -> np.ndarray:
    return emb_model.encode(text)

In [5]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """
    Calculate cosine similarity between two vectors.
    
    Args:
        a: First vector
        b: Second vector
        
    Returns:
        Cosine similarity score between -1 and 1
    """
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

In [6]:
query_key = 'อยากวิเคราะห์'
nextaction_key = 'ตัดสินใจ'

query_embed = embed_text(query_key)
nextaction_embed = embed_text(nextaction_key)

In [10]:
# เพิ่มข้อมูลทดสอบจาก query_test.py
from package.intent_hub.query_test import query_test
from package.intent_hub.nextaction_test import nextaction_test

# สร้าง model ใหม่ด้วย query_test
model_test = MultiIntentClassifier()
model_test.add_examples(intent="query", examples=query_general)
query_centroid = np.mean(model_test.vectors, axis=0)

# สร้าง nextaction centroid
model_nextaction = MultiIntentClassifier()
model_nextaction.add_examples(intent="nextaction", examples=nextaction)
nextaction_centroid = np.mean(model_nextaction.vectors, axis=0)

In [ ]:
# เปรียบเทียบ accuracy ระหว่าง keyword กับ centroid
def compare_methods():
    # ข้อมูลทดสอบ
    test_data = []
    for text in query_test:
        test_data.append({'text': text, 'true_intent': 'query'})
    for text in nextaction_test:
        test_data.append({'text': text, 'true_intent': 'nextaction'})
    
    keyword_correct = 0
    centroid_correct = 0
    
    for item in test_data:
        text_embed = embed_text(item['text'])
        
        # Method 1: Keyword-based
        query_sim_kw = cosine_similarity(text_embed, query_embed)
        # nextaction_sim_kw = cosine_similarity(text_embed, nextaction_embed)
        nextaction_sim_kw = np.linalg.norm(text_embed - nextaction_embed)
        pred_kw = "query" if query_sim_kw > nextaction_sim_kw else "nextaction"
        
        # Method 2: Centroid-based
        query_sim_ct = cosine_similarity(text_embed, query_centroid)
        # nextaction_sim_ct = cosine_similarity(text_embed, nextaction_centroid)
        nextaction_sim_ct = np.linalg.norm(text_embed - nextaction_embed)
        pred_ct = "query" if query_sim_ct > nextaction_sim_ct else "nextaction"
        
        if pred_kw == item['true_intent']:
            keyword_correct += 1
        if pred_ct == item['true_intent']:
            centroid_correct += 1
    
    total = len(test_data)
    print(f"Keyword method accuracy: {keyword_correct/total:.2%} ({keyword_correct}/{total})")
    print(f"Centroid method accuracy: {centroid_correct/total:.2%} ({centroid_correct}/{total})")

compare_methods()


Keyword method accuracy: 80.00% (56/70)
Centroid method accuracy: 97.14% (68/70)


In [14]:


def test_with_centroid():
    # ข้อมูลทดสอบ
    test_data = []
    for text in query_test:
        test_data.append({'text': text, 'true_intent': 'query'})
    for text in nextaction_test:
        test_data.append({'text': text, 'true_intent': 'nextaction'})
    
    correct = 0
    total = len(test_data)
    
    print("=== ทดสอบด้วย Query Test Centroid ===")
    
    for item in test_data:
        text_embed = embed_text(item['text'])
        
        query_sim = cosine_similarity(text_embed, query_test_centroid)
        nextaction_sim = cosine_similarity(text_embed, nextaction_centroid)
        
        predicted = "query" if query_sim > nextaction_sim else "nextaction"
        is_correct = predicted == item['true_intent']
        
        if is_correct:
            correct += 1
            
        print(f"{'✓' if is_correct else '✗'} True: {item['true_intent']} | Pred: {predicted}")
        print(f"  Query: {query_sim:.3f}, NextAction: {nextaction_sim:.3f}")
    
    accuracy = correct / total
    print(f"Accuracy: {accuracy:.2%} ({correct}/{total})")
    return accuracy

test_with_centroid()


=== ทดสอบด้วย Query Test Centroid ===
✓ True: query | Pred: query
  Query: 0.724, NextAction: 0.613
✓ True: query | Pred: query
  Query: 0.623, NextAction: 0.491
✓ True: query | Pred: query
  Query: 0.380, NextAction: 0.343
✓ True: query | Pred: query
  Query: 0.659, NextAction: 0.584
✓ True: query | Pred: query
  Query: 0.806, NextAction: 0.621
✓ True: query | Pred: query
  Query: 0.681, NextAction: 0.546
✓ True: query | Pred: query
  Query: 0.728, NextAction: 0.558
✓ True: query | Pred: query
  Query: 0.735, NextAction: 0.578
✓ True: query | Pred: query
  Query: 0.681, NextAction: 0.614
✓ True: query | Pred: query
  Query: 0.641, NextAction: 0.581
✓ True: query | Pred: query
  Query: 0.637, NextAction: 0.593
✓ True: query | Pred: query
  Query: 0.703, NextAction: 0.524
✓ True: query | Pred: query
  Query: 0.722, NextAction: 0.559
✓ True: query | Pred: query
  Query: 0.649, NextAction: 0.564
✓ True: query | Pred: query
  Query: 0.750, NextAction: 0.580
✓ True: query | Pred: query
  Qu

0.9714285714285714